# EDSR50 Baseline Evaluation Metrics

In [ ]:
import torch
import numpy as np
import time
import os
import glob
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import sys

# Ensure EDSR50 is in the path
sys.path.append(os.path.abspath('.'))
from src.model.edsr import MultiTaskEDSR
from src.config import Config

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Evaluating on {device}")

# Load the original EDSR50 Model
model = MultiTaskEDSR(
    scale=Config.SCALE,
    num_filters=Config.NUM_FILTERS,
    num_res_blocks=Config.NUM_RES_BLOCKS,
    channels=Config.CHANNELS
).to(device)

checkpoint_path = "checkpoints/best_model.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
model.eval()
print("EDSR50 Model loaded successfully!")


In [ ]:
# Discover all paired images
noisy_files = sorted(glob.glob("../train/NoisyLR/*.npy"))
gt_files = sorted(glob.glob("../train/GT/*.npy"))

print(f"Found {len(noisy_files)} paired test images for comprehensive evaluation.")


In [ ]:
inference_times = []
ssim_scores = []
psnr_scores = []
strictly_in_range = True
min_val = float('inf')
max_val = float('-inf')

print("Beginning evaluation over the entire dataset...")

with torch.no_grad():
    for i, (n_path, gt_path) in enumerate(zip(noisy_files, gt_files)):
        # 1. Load Data
        noisy_arr = np.load(n_path).astype(np.float32)
        gt_arr = np.load(gt_path).astype(np.float32)
        
        # 2. Format for EDSR50 (B, C, H, W)
        # EDSR50 natively uses 1-channel Grayscale inputs
        noisy_t = torch.from_numpy(noisy_arr).unsqueeze(0).unsqueeze(0).to(device)
        
        # 3. Synchronize GPU for accurate millisecond timing
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_time = time.time()
        
        # 4. Inference
        pred_t = model(noisy_t)
        
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end_time = time.time()
        
        inference_times.append(end_time - start_time)
        
        # 5. Extract output array
        # Squeeze out the batch and channel dimensions since they are 1
        pred_arr = pred_t.squeeze().cpu().numpy()
        
        # 6. Check Mathematical Bounds
        current_min = np.min(pred_arr)
        current_max = np.max(pred_arr)
        if current_min < min_val: min_val = current_min
        if current_max > max_val: max_val = current_max
        
        if current_min < 0.0 or current_max > 1.0:
            strictly_in_range = False
            
        # 7. Compute Scikit-Image Metrics (Standardized)
        ssim_val = ssim(gt_arr, pred_arr, data_range=1.0)
        psnr_val = psnr(gt_arr, pred_arr, data_range=1.0)
        
        ssim_scores.append(ssim_val)
        psnr_scores.append(psnr_val)

        if (i + 1) % 500 == 0:
            print(f"Processed {i + 1} / {len(noisy_files)} images...")

print("Evaluation Complete!")


In [ ]:
print("="*60)
print("1. END-TO-END INFERENCE TIME (GPU)")
print("="*60)
print(f"   Average Time   : {np.mean(inference_times)*1000:.2f} ms")
print(f"   Peak (Fastest) : {np.min(inference_times)*1000:.2f} ms")
print(f"   Worst (Slowest): {np.max(inference_times)*1000:.2f} ms")
print("\n" + "="*60)
print("2. IMAGE QUALITY METRICS (Scikit-Image)")
print("="*60)
print(f"   Average SSIM   : {np.mean(ssim_scores):.4f}")
print(f"   Highest SSIM   : {np.max(ssim_scores):.4f}")
print(f"   Lowest SSIM    : {np.min(ssim_scores):.4f}")
print(f"   Average PSNR   : {np.mean(psnr_scores):.2f} dB")
print("\n" + "="*60)
print("3. NUMERICAL BOUNDS")
print("="*60)
print(f"   Global Minimum Value : {min_val:.6f}")
print(f"   Global Maximum Value : {max_val:.6f}")
if strictly_in_range:
    print("   [PASS] All output array values are STRICTLY within [0.0, 1.0].")
else:
    print("   [FAIL] Output array values are NOT strictly within [0.0, 1.0].")
print("="*60)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fig, axs = plt.subplots(3, 1, figsize=(15, 12))
window = 50  # 50-image moving average

ssim_smooth = pd.Series(ssim_scores).rolling(window=window, min_periods=1).mean()
psnr_smooth = pd.Series(psnr_scores).rolling(window=window, min_periods=1).mean()
inference_times_ms = [t * 1000 for t in inference_times]
time_smooth = pd.Series(inference_times_ms).rolling(window=window, min_periods=1).mean()

# 1. SSIM Graph
axs[0].plot(ssim_scores, color='blue', alpha=0.15, label='Raw SSIM')
axs[0].plot(ssim_smooth, color='darkblue', linewidth=2, label=f'{window}-Image Moving Avg')
axs[0].set_title('SSIM over 3200 Images', fontsize=14)
axs[0].set_ylabel('SSIM')
axs[0].grid(True, linestyle='--', alpha=0.6)
axs[0].legend(loc='lower right')

# 2. PSNR Graph
axs[1].plot(psnr_scores, color='green', alpha=0.15, label='Raw PSNR')
axs[1].plot(psnr_smooth, color='darkgreen', linewidth=2, label=f'{window}-Image Moving Avg')
axs[1].set_title('PSNR over 3200 Images', fontsize=14)
axs[1].set_ylabel('PSNR (dB)')
axs[1].grid(True, linestyle='--', alpha=0.6)
axs[1].legend(loc='lower right')

# 3. Inference Time Graph
axs[2].plot(inference_times_ms, color='red', alpha=0.15, label='Raw Time')
axs[2].plot(time_smooth, color='darkred', linewidth=2, label=f'{window}-Image Moving Avg')
axs[2].set_title('Inference Time over 3200 Images', fontsize=14)
axs[2].set_ylabel('Time (ms)')
axs[2].set_xlabel('Image Index', fontsize=12)
axs[2].set_ylim([0, 300])
axs[2].grid(True, linestyle='--', alpha=0.6)
axs[2].legend(loc='upper right')

plt.tight_layout()
plt.show()
